# `EventData: TypedDict(total=False)`

`EventData` stores the payload associated with a standard Runnable streaming event.

Every field is optional because the available payload depends on whether the event represents a start, stream, end, or error.

## Fields

```python
input: Any # Input passed to the Runnable that generated the event
error: NotRequired[BaseException] # Exception raised during Runnable execution
output: Any # Final output produced by the Runnable
chunk: Any # Output chunk emitted while the Runnable is streaming
tool_call_id: NotRequired[str | None] # Tool-call identifier associated with a tool error
```

## Field Availability

- `input` may appear at the start or end of execution.
- Streaming Runnables may not expose the complete input until execution ends.
- `error` is available only when execution raises an exception.
- `output` is normally available only in an end event.
- `chunk` is normally available only in a stream event.
- Adding compatible chunks should generally reconstruct the final output.
- `tool_call_id` is used by `on_tool_error` events to associate an error with a tool call.


In [ ]:
from langchain_core.runnables.schema import EventData # Import the EventData TypedDict

start_data: EventData = { # Create data for a Runnable start event
    "input": "hello langchain", # Store the input passed to the Runnable
} # Finish creating the start-event data

stream_data: EventData = { # Create data for a Runnable stream event
    "chunk": "HELLO ", # Store one output chunk produced during streaming
} # Finish creating the stream-event data

end_data: EventData = { # Create data for a Runnable end event
    "input": "hello langchain", # Store the original Runnable input
    "output": "HELLO LANGCHAIN", # Store the final Runnable output
} # Finish creating the end-event data

error_data: EventData = { # Create data for a Runnable error event
    "input": "invalid input", # Store the input that caused the error
    "error": ValueError("Input processing failed"), # Store the raised exception
    "tool_call_id": "tool-call-101", # Associate the error with a tool call
} # Finish creating the error-event data

print("Start data:", start_data) # Display the start-event payload

print("Stream data:", stream_data) # Display the stream-event payload

print("End data:", end_data) # Display the end-event payload

print("Error data:", error_data) # Display the error-event payload

# `BaseStreamEvent: TypedDict`

`BaseStreamEvent` defines the fields shared by standard and custom events produced through Runnable event streaming.

## Fields

```python
event: str # Event name describing the Runnable type and execution stage
run_id: str # Unique identifier for the Runnable execution
tags: NotRequired[list[str]] # Tags associated with the Runnable execution
metadata: NotRequired[dict[str, Any]] # Metadata associated with the Runnable execution
parent_ids: Sequence[str] # Parent run identifiers from the root parent to the immediate parent
```

## Event Name Format

```text
on_[runnable_type]_[start|stream|end]
```

Common Runnable event types include:

```text
llm
chat_model
prompt
tool
chain
```

## Parent Run Behaviour

- Root events contain an empty `parent_ids` sequence.
- Child events contain the identifiers of their parent executions.
- Parent identifiers are ordered from the root parent to the immediate parent.
- Event schema version `v2` provides parent identifiers.
- Event schema version `v1` returns an empty parent list.


In [ ]:
from uuid import uuid4 # Import UUID generator

from langchain_core.runnables.schema import BaseStreamEvent # Import the base stream-event type

root_run_id: str = str(uuid4()) # Generate an identifier for the root Runnable execution

child_run_id: str = str(uuid4()) # Generate an identifier for the child Runnable execution

root_event: BaseStreamEvent = { # Create a root-level stream event
    "event": "on_chain_start", # Describe the Runnable type and execution stage
    "run_id": root_run_id, # Store the root execution identifier
    "tags": ["main-chain"], # Add optional tracing tags
    "metadata": {"source": "jupyter"}, # Add optional event metadata
    "parent_ids": [], # Use an empty list because the event has no parent
} # Finish creating the root event

child_event: BaseStreamEvent = { # Create a child-level stream event
    "event": "on_tool_end", # Describe the child Runnable event
    "run_id": child_run_id, # Store the child execution identifier
    "tags": ["calculator-tool"], # Add optional child-event tags
    "metadata": {"tool": "calculator"}, # Add optional child-event metadata
    "parent_ids": [root_run_id], # Store the root execution as the child's parent
} # Finish creating the child event

print("Root event:", root_event) # Display the root event

print("Child event:", child_event) # Display the child event

# `StandardStreamEvent: BaseStreamEvent`

`StandardStreamEvent` represents a normal LangChain lifecycle event.

## Fields

```python
event: str # Standard Runnable lifecycle event name
run_id: str # Unique identifier for the Runnable execution
tags: NotRequired[list[str]] # Tags associated with the Runnable execution
metadata: NotRequired[dict[str, Any]] # Metadata associated with the Runnable execution
parent_ids: Sequence[str] # Parent run identifiers
data: EventData # Payload associated with the lifecycle event
name: str # Name of the Runnable that generated the event
```

The contents of `data` depend on whether the event represents a start, stream, end, or error.

In [ ]:
from uuid import uuid4 # Import UUID generator

from langchain_core.runnables.schema import StandardStreamEvent # Import the standard stream-event type

run_id: str = str(uuid4()) # Generate one execution identifier

start_event: StandardStreamEvent = { # Create a Runnable start event
    "event": "on_chain_start", # Set the lifecycle event name
    "run_id": run_id, # Store the execution identifier
    "name": "uppercase_chain", # Store the Runnable name
    "tags": ["text-processing"], # Add optional tracing tags
    "metadata": {"source": "jupyter"}, # Add optional execution metadata
    "parent_ids": [], # Use an empty list because this is a root run
    "data": { # Store the start-event payload
        "input": "hello langchain", # Store the input passed to the Runnable
    }, # Finish the payload
} # Finish creating the start event

stream_event: StandardStreamEvent = { # Create a Runnable streaming event
    "event": "on_chain_stream", # Set the streaming lifecycle event name
    "run_id": run_id, # Reuse the same execution identifier
    "name": "uppercase_chain", # Store the Runnable name
    "tags": ["text-processing"], # Add optional tracing tags
    "metadata": {"source": "jupyter"}, # Add optional execution metadata
    "parent_ids": [], # Keep the event at the root level
    "data": { # Store the streaming-event payload
        "chunk": "HELLO LANGCHAIN", # Store the emitted output chunk
    }, # Finish the payload
} # Finish creating the stream event

end_event: StandardStreamEvent = { # Create a Runnable end event
    "event": "on_chain_end", # Set the completion lifecycle event name
    "run_id": run_id, # Reuse the same execution identifier
    "name": "uppercase_chain", # Store the Runnable name
    "tags": ["text-processing"], # Add optional tracing tags
    "metadata": {"source": "jupyter"}, # Add optional execution metadata
    "parent_ids": [], # Keep the event at the root level
    "data": { # Store the end-event payload
        "input": "hello langchain", # Store the original input
        "output": "HELLO LANGCHAIN", # Store the final output
    }, # Finish the payload
} # Finish creating the end event

events: list[StandardStreamEvent] = [ # Store the lifecycle events in order
    start_event, # Add the start event
    stream_event, # Add the stream event
    end_event, # Add the end event
] # Finish creating the event list

for event in events: # Iterate through the events
    print(event["event"], "->", event["data"]) # Display each event name and payload

# `CustomStreamEvent: BaseStreamEvent`

`CustomStreamEvent` represents an event explicitly dispatched by application code.

## Fields

```python
event: Literal["on_custom_event"] # Fixed event type used for custom events
run_id: str # Unique identifier for the Runnable execution
tags: NotRequired[list[str]] # Tags associated with the Runnable execution
metadata: NotRequired[dict[str, Any]] # Metadata associated with the Runnable execution
parent_ids: Sequence[str] # Parent run identifiers
name: str # User-defined custom event name
data: Any # Free-form custom event payload
```

Unlike standard event data, custom event payloads may contain any value.


In [ ]:
from uuid import uuid4 # Import the UUID generator

from langchain_core.runnables.schema import CustomStreamEvent # Import the custom stream-event type

parent_run_id: str = str(uuid4()) # Generate an identifier for the parent Runnable

custom_event: CustomStreamEvent = { # Create a user-defined stream event
    "event": "on_custom_event", # Use the fixed custom-event type
    "run_id": str(uuid4()), # Generate an identifier for this event execution
    "name": "progress_update", # Set the custom event name
    "tags": ["data-processing"], # Add optional tracing tags
    "metadata": {"source": "jupyter"}, # Add optional event metadata
    "parent_ids": [parent_run_id], # Store the parent Runnable identifier
    "data": { # Add any custom payload
        "message": "Processing completed", # Store a progress message
        "percentage": 100, # Store a numeric progress value
        "successful": True, # Store a Boolean result
    }, # Finish defining the custom payload
} # Finish creating the custom event

print("Event type:", custom_event["event"]) # Display the fixed event type

print("Event name:", custom_event["name"]) # Display the custom event name

print("Event data:", custom_event["data"]) # Display the free-form payload

# `StreamEvent`

`StreamEvent` represents either a standard Runnable event or a user-defined custom event.

```python
StreamEvent = StandardStreamEvent | CustomStreamEvent # Union of supported stream event schemas
```

## Developer-Facing Top-Level Statements

```python
EventData # Payload schema for standard stream events
BaseStreamEvent # Shared stream-event fields
StandardStreamEvent # Standard Runnable lifecycle event
CustomStreamEvent # User-defined custom event
StreamEvent # Union of standard and custom events
```
No public functions are defined in this module.